<a href="https://colab.research.google.com/github/CUHK-DH-Lab/YCRG_Cross-Cultural_Analytics/blob/main/DR_week_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
pip install numpy pandas scikit-learn requests

In [9]:
import re
import requests
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

GITHUB_API = "https://api.github.com/repos/cltk/lat_text_latin_library/contents"

LATIN_WORLDS = {
    "republican": ["caesar", "cicero", "sallust"],
    "augustan": ["vergil", "horace", "ovid"],
    "silver": ["tacitus", "seneca"],
    "late_antique": ["augustine", "jerome"]
}

HEADERS = {"Accept": "application/vnd.github.v3+json"}

# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

def normalize_latin(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def list_txt_files(author):
    """
    Ask GitHub what files actually exist for this author.
    """
    url = f"{GITHUB_API}/{author}"
    r = requests.get(url, headers=HEADERS)
    r.raise_for_status()
    return [
        f["download_url"]
        for f in r.json()
        if f["name"].endswith(".txt")
    ]

def download_author_corpus(author):
    """
    Download and concatenate all Latin .txt files for an author.
    """
    urls = list_txt_files(author)
    texts = []
    for url in urls:
        r = requests.get(url)
        r.raise_for_status()
        texts.append(r.text)
    return normalize_latin("\n".join(texts))

# ------------------------------------------------------------
# 3. Download real Latin corpora
# ------------------------------------------------------------

corpus_texts = []
corpus_authors = []
corpus_periods = []

print("Downloading authentic Latin texts (via GitHub API)...\n")

for period, authors in LATIN_WORLDS.items():
    for author in authors:
        try:
            text = download_author_corpus(author)
            corpus_texts.append(text)
            corpus_authors.append(author)
            corpus_periods.append(period)
            print(f"✓ {author} ({period})")
        except Exception as e:
            print(f"✗ Skipped {author}: {e}")

print("\nCorpus download complete.\n")

# ------------------------------------------------------------
# 4. Build stylometric model (character n-grams)
# ------------------------------------------------------------

vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.9
)

X_corpus = vectorizer.fit_transform(corpus_texts)

# ------------------------------------------------------------
# 5. Load your texts
# ------------------------------------------------------------

with open("RICCI.txt", encoding="utf-8") as f:
    ricci_text = normalize_latin(f.read())

with open("MISSIONE.txt", encoding="utf-8") as f:
    missione_text = normalize_latin(f.read())

# ------------------------------------------------------------
# 6. Comparison function
# ------------------------------------------------------------

def compare_text(text, label):
    X_text = vectorizer.transform([text])
    sims = cosine_similarity(X_text, X_corpus)[0]

    df = pd.DataFrame({
        "author": corpus_authors,
        "period": corpus_periods,
        "similarity": sims
    }).sort_values("similarity", ascending=False)

    print(f"\n{label} — Closest authors:")
    print(df.head(10))

    print(f"\n{label} — Closest Latin worlds (by period):")
    print(
        df.groupby("period")["similarity"]
        .mean()
        .sort_values(ascending=False)
    )

# ------------------------------------------------------------
# 7. Run analysis
# ------------------------------------------------------------

compare_text(ricci_text, "RICCI")
compare_text(missione_text, "MISSIONE")


✓ caesar (republican)
✓ cicero (republican)
✗ Skipped sallust: 404 Client Error: Not Found for url: https://api.github.com/repos/cltk/lat_text_latin_library/contents/sallust
✓ vergil (augustan)
✓ horace (augustan)
✓ ovid (augustan)
✓ tacitus (silver)
✗ Skipped seneca: 404 Client Error: Not Found for url: https://api.github.com/repos/cltk/lat_text_latin_library/contents/seneca
✓ augustine (late_antique)
✓ jerome (late_antique)

Corpus download complete.


RICCI — Closest authors:
      author        period  similarity
1     cicero    republican    0.479869
5    tacitus        silver    0.368829
0     caesar    republican    0.310496
6  augustine  late_antique    0.148385
4       ovid      augustan    0.135083
7     jerome  late_antique    0.128254
3     horace      augustan    0.099823
2     vergil      augustan    0.071454

RICCI — Closest Latin worlds (by period):
period
republican      0.395182
silver          0.368829
late_antique    0.138319
augustan        0.102120
Name: similari

Let's use Google Vertext AI to do the same thing...

In [ ]:
!pip install -U google-genai
import os
import json
from google import genai

In [3]:
# ... (Keep your existing imports and environment setup) ...

def classify(name, text):
    if not text:
        print(f"File for {name} not found.")
        return

    # Moving the "role" to a system instruction for better results
    system_instruction = "You are a Latin philologist. Answer with the work title ONLY. Do not explain."

    prompt = f"Which ONE of these works is this text most similar to?\n{WORKS}\n\nText ({name}):\n{text}"

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config={
            "system_instruction": system_instruction,
            "temperature": 0.0,
            "max_output_tokens": 100  # Increased to prevent truncation
        }
    )

    if response.text:
        # We take only the first line in case it still adds extra text
        answer = response.text.strip().split('\n')[0]
        print(f"{name} → {answer}")
    else:
        print(f"{name} → Error: No text returned. Finish reason: {response.candidates[0].finish_reason}")

# --- Execute ---
classify("RICCI", ricci)
classify("MISSIONE", missione)


RICCI → Augustine
MISSIONE → Jerome – Ep
